# 2D guillotine 切割 cgcut1 —— 对偶/割推导总览（单张板材模式结构）

问题：$\max\{\sum v_i a_i:\ a\in P\}$，$P$=guillotine 可行模式集（固定朝向，完整池 1748 模式）。
最优 244（文献一致；LP = IP）。

## 01 直接建模 —— 无对偶
候选全枚举（2119 计数向量）+ 精确 guillotine 检验递归（切分归纳 + 记忆化）⇒ 池完整 ⇒ 池 IP = 精确解 244。

## 02 列生成 —— LP 即最大模式价值
主问题 = 模式凸包 LP：$\max\sum_p val_p\lambda_p$ s.t. $\sum\lambda_p=1$——线性目标在单纯形上的
最大 = 最大系数 = 244 = IP。定价 = 池扫描（池完整 ⇒ 等价动态定价）。

## 03 Benders（max 版本）—— SP 对偶 → 上界割
SP(y)：$\max\sum\lambda_p val_p$，$\sum\lambda=1$，$\lambda\le y$；对偶 $\min\theta+\sum\sigma_p y_p$
s.t. $\theta+\sigma_p\ge val_p$。取 $\sigma_p=\max(0,val_p-\theta^*)$ ⇒ 割
$\theta\le V(y^k)+\sum\sigma_p(y_p-y^k_p)$（对任意 y 有效）；主问题 $\max\theta$ 收敛到 $\max_p val_p=244$。

## 04 拉格朗日 —— 松弛件数上限
$L(\mu)=\sum\mu_i q_i+\max_{a\in P}\sum(v_i-\mu_i)a_i$（子问题=池扫描）；次梯度 $g=q-a^*$，
下降 $\mu\leftarrow\mu-\alpha g$；$\min L$ = LP = 244（strong duality）。

## 05 LBBD —— 逻辑割
主问题=候选最大化（无几何）；子问题=精确 guillotine 检验；不可切 → no-good 排除该向量
（$|a-c|\ge1$ 线性化）。首轮排除 (2,1,3,0,1,1,1)，收敛到可行 244。

## 07 Branch-and-Price —— 件数分支
根 LP = 244 整数（单模式）→ 0 分支证明最优；分支规则（$a_i\le k$ / $a_i\ge k+1$ + 节点池过滤）已实现。


In [1]:
# 数值验证：池 IP = LP(conv) = 244
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="numpy")
import sys, platform
import ortools
sys.path.insert(0, "/mnt/d/exactTest/column-generation-solvers/cutting_2d_cgcut/scripts")
import cg2_core as cc
from ortools.math_opt.python import mathopt
print("python", platform.python_version(), "| ortools", ortools.__version__)

term, obj, sel, wt = cc.pool_ip()
lp = cc.conv_lp()
print(f"完整池 {cc.P} 模式 | 池 IP = {obj}（模式 {cc.POOL[sel[0]] if sel else None}）")
print(f"LP(conv) = {round(lp,4)} | LP = IP = 244: {abs(lp-244)<1e-9 and abs(obj-244)<1e-9}")
print(f"最优模式价值复算: {sum(cc.V[i]*cc.POOL[sel[0]][i] for i in range(cc.M))}")


m=7 | 板材 15x10 | items [(8, 4, 2, 66), (3, 7, 1, 35), (8, 2, 3, 24), (3, 4, 5, 17), (3, 3, 2, 11), (3, 2, 2, 8), (2, 1, 1, 2)] | 旋转=False（文献约定） | 文献最优 244
加载缓存的固定朝向完整池: 1748 模式
python 3.10.20 | ortools 9.15.6755
完整池 1748 模式 | 池 IP = 244.0（模式 (2, 1, 1, 2, 1, 1, 0)）
LP(conv) = 244.0 | LP = IP = 244: True
最优模式价值复算: 244
